# 06 - Multi-IDF, multi-EPW, and custom plan

This notebook demonstrates multi-IDF and multi-EPW execution with a custom plan and callable-based outputs.

The plan simulates the two sample models (high and low performance) with the two weather files and two thicknesses of the roof material, a BESOS parameter added to the accim parameters, so the results show the effect of each one.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `ParametricSimulation`
- `return_time_series`
- `ParametricSimulation.set_output_readers`
- `ParametricSimulation.run_parametric_simulation`
- `WorkspaceArtifactCleaner`
- `ef.get_building`
- `parametric.discover_available_outputs`
- `parametric.select_outputs`
- `parametric.apply_outputs_preflight`
- `parametric.set_output_readers`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## Multi-IDF and multi-EPW: import workflow dependencies

We start by importing the APIs required for this notebook block.

Keeping imports explicit at the top of each topic notebook makes selective reruns safer when you jump between sections of the series.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

`PROCESSES` is the number of simulations run in parallel: half of the logical CPUs by default.

In [ ]:
import os
from pathlib import Path

import accim
import pandas as pd
from besos import eppy_funcs as ef
from besos.parameters import FieldSelector, Parameter, RangeParameter
from accim.utils import WorkspaceArtifactCleaner
from accim.parametric_and_optimisation.main import ParametricSimulation
from accim.parametric_and_optimisation.objectives import return_time_series

pd.set_option("display.max_columns", 200)

# Simulations run in parallel: half of the logical CPUs.
PROCESSES = max(1, (os.cpu_count() or 2) // 2)


## Multi-IDF and multi-EPW: set project paths and discover IDF/EPW files

Here we declare the baseline project assets (IDFs, EPWs, and output folder).

For Multi-IDF and multi-EPW, this is the configuration anchor. If asset names change, update this block before touching execution cells.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `idf_list`: reused in subsequent workflow blocks.
- `epw_list`: reused in subsequent workflow blocks.
- `RESULTS_DIRNAME`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `PROJECT_ROOT`, `idf_list`, `epw_list`, `RESULTS_DIRNAME`.

In [ ]:
PROJECT_ROOT = Path.cwd()

idf_list = [
	"OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance.idf",
	"OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_low_performance.idf",
]
epw_list = ["Seville.epw", "Sydney.epw"]
RESULTS_DIRNAME = "nb06_multi_idf_multi_epw_custom_results"

print("IDF files found:", [(PROJECT_ROOT / idf).exists() for idf in idf_list])
print("EPW files found:", [(PROJECT_ROOT / epw).exists() for epw in epw_list])


## Multi-IDF and multi-EPW: capture baseline workspace state for cleanup

We initialize artifact tracking so generated files can be reviewed and cleaned in a controlled way.

This is especially useful in iterative notebook sessions where repeated runs can quickly accumulate temporary folders and CSV files.

Key arguments used in this cell and why they matter:
- `workspace_root`: defines this setting for the current call.

Variables created in this cell:
- `artifact_cleaner`: reused in subsequent workflow blocks.
- `baseline_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `artifact_cleaner`, `baseline_files`.

In [ ]:
artifact_cleaner = WorkspaceArtifactCleaner(workspace_root=PROJECT_ROOT)
baseline_files = artifact_cleaner.capture_initial_state()
print("Baseline file count:", len(baseline_files))


## Multi-IDF and multi-EPW: load the IDF models

Load every model of the study.

Both sample models have the same geometry and differ in their envelope: for example, the roof material conducts 0.008 W/m-K in the high-performance model and 0.2 W/m-K in the low-performance one. Each model is simulated from its own copy, with the settings applied to it below.

Variables created in this cell:
- `buildings`: one IDF object per file of `idf_list`.

What to verify right after running this cell:
- Confirm that both models are loaded.

In [ ]:
buildings = [ef.get_building(idf) for idf in idf_list]
print("Models loaded:", len(buildings))


## Multi-IDF and multi-EPW: apply runtime-reduction settings

Apply runtime reductions to keep notebook runs faster during exploration.

This block shortens exploratory runs by reducing simulation burden. Use it for prototyping and switch to fuller settings when producing final results.

`reduce_runtime` also changes the run period, to 1 January by default, so its run period arguments keep the full year: the results stay annual. `Run_Simulation_for_Sizing_Periods = "No"` keeps the design days of the sample model out of the annual results (they are still used to size the HVAC system).

Key arguments used in this cell and why they matter:
- `idf_object`: each loaded model.
- `timesteps`: timesteps per hour.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
for building in buildings:
    accim.utils.reduce_runtime(
        idf_object=building,
        timesteps=2,
        runperiod_begin_month=1,
        runperiod_begin_day_of_month=1,
        runperiod_end_month=12,
        runperiod_end_day_of_month=31,
    )
    building.idfobjects["SimulationControl"][0].Run_Simulation_for_Sizing_Periods = "No"


## Multi-IDF and multi-EPW: initialize parametric simulation object

Create the parametric simulation object with weather context and model settings.

This creates the parametric simulation object that will orchestrate outputs, parameters, and run plans. From here on, the workflow uses this object as the main controller.

Key arguments used in this cell and why they matter:
- `buildings`: the models of the study. The `idf` column of the plan says which one each row uses; a plan without that column is simulated with every model.
- `parameters_type`: parameter family used in the simulation setup.
- `epws`: list of EPW weather files used to run scenarios.
- `output_freqs`: time frequency requested for outputs.

Variables created in this cell:
- `parametric`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `parametric`.

In [ ]:
parametric = ParametricSimulation(
	buildings=buildings,
	parameters_type="accim custom model",
	epws=epw_list,
	output_freqs=["hourly"],
)


## Multi-IDF and multi-EPW: discover available meter and variable outputs

Inspect available meters and variables before selecting valid outputs.

Before selecting outputs, we inspect what meters and variables are currently available. That discovery step reduces mistakes when requesting objective or analysis outputs.

Key arguments used in this cell and why they matter:
- `reduce_sim_time`: enables a fast scan mode when discovering outputs.
- `prefer`: preferred backend for output discovery.

Variables created in this cell:
- `df_meters_available`: reused in subsequent workflow blocks.
- `df_vars_available`: reused in subsequent workflow blocks.
- `outputs_meta`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check that the discovered meter/variable tables are not unexpectedly empty.
- Confirm the reported discovery source matches your preferred backend.

In [ ]:
available_outputs = parametric.discover_available_outputs(
	reduce_sim_time=True,
	prefer="testsimeplus",
)
df_meters_available = available_outputs["meters"]
df_vars_available = available_outputs["variables"]
outputs_meta = available_outputs["meta"]

print("Discovery source:", outputs_meta.get("source"))
print("Meters available:", len(df_meters_available))
print("Variables available:", len(df_vars_available))


## Multi-IDF and multi-EPW: validate and apply output configuration

Select target outputs and validate them before applying changes to the model.

This section applies explicit output selection and preflight validation. Validating before execution keeps result schemas stable across runs.

Key arguments used in this cell and why they matter:
- `meters`: energy meters to include as outputs.
- `match`: matching rule used to select outputs.
- `on_missing`: behavior when a requested output is missing.
- `suggest`: shows suggestions when no exact match is found.
- `variables`: output variables to include by name or pattern.
- `df_meters_sel`: table of selected meters to apply to the IDF.
- `df_vars_sel`: table of selected variables to apply to the IDF.
- `clean_mode`: cleanup mode applied before adding outputs.
- `validate_before_apply`: whether to validate before editing the IDF.
- `validate_after_apply`: whether to validate after editing the IDF.
- `df_output_meter`: final meter table used by the simulation.
- `df_output_variable`: final variable table used by the simulation.

Variables created in this cell:
- `objective_meters`: reused in subsequent workflow blocks.
- `df_meters_sel`: reused in subsequent workflow blocks.
- `_`: reused in subsequent workflow blocks.
- `df_vars_sel`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [ ]:

objective_meters = ["Heating:Electricity"]
selection = parametric.select_outputs(
    meters=objective_meters,
    match="case_insensitive",
    on_missing="raise",
    suggest=True,
)
df_meters_sel = selection["meters"]

selection = parametric.select_outputs(
    variables=["Running Average Outdoor Air Temperature"],
    match="contains",
    on_missing="raise",
    suggest=True,
)
df_vars_sel = selection["variables"]

df_vars_sel = df_vars_sel.copy().head(1)
df_vars_sel["name"] = df_vars_sel["variable_name"] + "_time series"
df_vars_sel["func"] = return_time_series

parametric.apply_outputs_preflight(
    df_meters_sel=df_meters_sel,
    df_vars_sel=df_vars_sel,
    clean_mode="all",
    validate_before_apply=False,
    validate_after_apply=True,
    on_missing="raise",
)
parametric.set_output_readers(
    df_output_meter=df_meters_sel,
    df_output_variable=df_vars_sel,
)

print("Selected meters:", list(df_meters_sel["key_name"]))
print("Selected variable output names:", list(df_vars_sel["name"]))


## Multi-IDF and multi-EPW: define decision variables and objective directions

Set parameter ranges or values and build the study problem.

Here we define the decision space and objective orientation used in simulation or optimization. These choices directly determine how the final result space will look.

accim writes its own parameters into the EMS programs of each model; the BESOS parameters of `additional_params` are written by BESOS in each simulation, as in an optimisation. With several processes, they are sent to each process, so their selectors must be picklable (`FieldSelector` is).

Key arguments used in this cell and why they matter:
- `accis_params_dict`: parameters of the custom model and their ranges.
- `additional_params`: other BESOS parameters, here the thickness of the roof material.
- `selector`: what the parameter changes in the IDF: the field `field_name` of the object `object_name` of the class `class_name`.
- `value_descriptors`: the name of the parameter and its range (`min_val`, `max_val`), in m.

Variables created in this cell:
- `accis_parameters`: reused in subsequent workflow blocks.
- `insulation_thickness_param`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `accis_parameters`, `insulation_thickness_param`.

In [ ]:
accis_parameters = {
	"CustAST_m": (0.2, 0.8),
	"CustAST_n": (10, 20),
	"CustAST_ASToffset": (2.0, 4.0),
	"CustAST_ASTall": (9, 14),
	"CustAST_ASTaul": (30, 36),
}

insulation_thickness_param = Parameter(
	selector=FieldSelector(
		class_name="Material",
		object_name="Residential Roof Construction Material",
		field_name="Thickness",
	),
	value_descriptors=RangeParameter(min_val=0.02, max_val=0.20, name="insulation_thickness_m"),
)

parametric.set_parameters(
	accis_params_dict=accis_parameters,
	additional_params=[insulation_thickness_param],
)
parametric.set_problem()

print("Problem inputs:", parametric.problem.names("inputs"))
print("Problem outputs:", parametric.problem.names("outputs"))


## Multi-IDF and multi-EPW: assemble the custom plan

The plan is a dataframe with one row per simulation: the value of each parameter, and the IDF and EPW of the row. The IDF can be named by its file name or by its identifier (the file name without `.idf`), and it must be one of the loaded models.

All the rows use the same comfort model, the ASHRAE 55 adaptive model with 80 % acceptability (m = 0.31, n = 17.8 °C, ±3.5 K, limits of 10 and 33.5 °C), so the results only change with the model, the climate and the roof thickness (0.04 or 0.16 m): 2 x 2 x 2 = 8 simulations.

Variables created in this cell:
- `comfort_model`: values of the parameters of the custom model.
- `custom_plan`: the plan.
- `plan_df`: the plan stored by `sampling_custom`, used below.

What to verify right after running this cell:
- Confirm that each combination of model, weather file and thickness appears once.

In [ ]:
comfort_model = {
    "CustAST_m": 0.31,
    "CustAST_n": 17.8,
    "CustAST_ASToffset": 3.5,
    "CustAST_ASTall": 10,
    "CustAST_ASTaul": 33.5,
}
custom_plan = pd.DataFrame(
    [
        {**comfort_model, "insulation_thickness_m": thickness, "idf": idf, "epw": epw}
        for idf in idf_list
        for epw in epw_list
        for thickness in (0.04, 0.16)
    ]
)

parametric.sampling_custom(custom_plan)
plan_df = parametric.parameters_values_df.copy()

print("Custom plan rows:", len(plan_df))
plan_df[["idf", "epw", "insulation_thickness_m"]]


## Multi-IDF and multi-EPW: estimate case count from plan and EPW multipliers

Before execution, we estimate workload size from plan cardinality and solver budget settings.

Use this checkpoint to align runtime expectations with available compute resources before launching.

Variables created in this cell:
- `epw_multiplier`: reused in subsequent workflow blocks.
- `idf_multiplier`: reused in subsequent workflow blocks.
- `estimated_simulations`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare estimated run count with your available compute/time budget.
- Adjust evaluations, population size, or climate count if the estimate is too high.

In [ ]:
epw_multiplier = 1 if "epw" in plan_df.columns else len(epw_list)
idf_multiplier = 1 if "idf" in plan_df.columns else len(parametric.buildings)
estimated_simulations = len(plan_df) * epw_multiplier * idf_multiplier

print("Plan rows:", len(plan_df))
print("EPW multiplier:", epw_multiplier)
print("IDF multiplier:", idf_multiplier)
print("Estimated simulation cases:", estimated_simulations)


## Multi-IDF and multi-EPW: execute parametric run and inspect output rows

This is the execution checkpoint for the configured parametric plan.

After the run, verify row counts and core objective columns immediately so issues are caught before post-processing.

The time-series output holds 8,760 values per row: the CSV, JSON and pickle files of the results keep them, while the Excel copy truncates such long cells, hence the warnings when the results are saved.

Key arguments used in this cell and why they matter:
- `out_dir`: output folder for generated artifacts and results.
- `df`: case plan dataframe executed in this run.
- `processes`: number of parallel processes for simulation runs.
- `keep_input`: whether generated input files are preserved.
- `keep_dirs`: whether per-run directories are preserved.

Variables created in this cell:
- `outputs`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare returned row count with the expected number of plan combinations.
- Inspect the first rows to ensure objective columns are present and populated.

In [ ]:

outputs = parametric.run_parametric_simulation(
    out_dir=RESULTS_DIRNAME,
    df=plan_df,
    processes=PROCESSES,
    keep_input=True,
    keep_dirs=False,
)
print("Rows:", len(outputs))
print(outputs.head())


### Reading the result

Each row of the results reports the IDF and EPW it was simulated with, in the order of the plan. The next cell summarises the heating electricity of each model and climate with each roof thickness.

In [ ]:
summary = outputs.assign(
    model=outputs["idf"].str.rsplit("_v2520_", n=1).str[-1],
    heating_kWh=outputs["Heating:Electricity"] / 3.6e6,
)
summary.pivot_table(index=["model", "epw"], columns="insulation_thickness_m", values="heating_kWh").round(0)


### Reading the result

Compare the columns to see the effect of the roof thickness in each model and climate, and the rows to see the effect of the model and the climate. The thicker roof material lowers the heating much more in the low-performance model, whose roof material conducts 25 times more heat (0.2 against 0.008 W/m-K).

## Multi-IDF and multi-EPW: inspect callable-derived output columns

Inspect outputs produced by callable-based columns and validate sample values.

Here we inspect columns generated through callable output functions to confirm they were created correctly. A quick type/value check helps detect schema issues early.

Variables created in this cell:
- `callable_cols`: reused in subsequent workflow blocks.
- `sample_col`: reused in subsequent workflow blocks.
- `sample_val`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm callable-derived columns exist and contain values with the expected type.
- Review a sample row to ensure callable outputs are aligned with source variables.

In [ ]:

callable_cols = [c for c in outputs.columns if c.endswith("_time series")]
print("Callable-derived columns:", callable_cols)

if callable_cols:
    sample_col = callable_cols[0]
    sample_val = outputs[sample_col].iloc[0]
    print("Sample column:", sample_col)
    print("Sample value type:", type(sample_val))


## Multi-IDF and multi-EPW: inspect generated artifacts before cleanup

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `max_items`: maximum number of items shown in logs.

Variables created in this cell:
- `generated_files`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `generated_files`.

In [ ]:
generated_files = artifact_cleaner.print_generated_files(max_items=200)


## Multi-IDF and multi-EPW: preview cleanup actions with dry-run

This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
artifact_cleaner.delete_generated_files(
	allow_patterns=[f"{RESULTS_DIRNAME}/**"],
	deny_patterns=[],
	dry_run=True,
	remove_empty_dirs=True,
)


This section audits generated artifacts and applies the configured cleanup policy.

Keeping cleanup explicit avoids accidental deletion while still preventing accumulation of stale run artifacts.

`DELETE_RESULTS` is False by default, so nothing is deleted. Set it to True to delete the files listed above.

Key arguments used in this cell and why they matter:
- `allow_patterns`: file patterns that are allowed for cleanup.
- `deny_patterns`: file patterns protected from cleanup.
- `dry_run`: whether cleanup only reports actions without deleting.
- `remove_empty_dirs`: whether empty directories are removed after cleanup.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
DELETE_RESULTS = False

if DELETE_RESULTS:
    artifact_cleaner.delete_generated_files(
        allow_patterns=[f"{RESULTS_DIRNAME}/**"],
        deny_patterns=[],
        dry_run=False,
        remove_empty_dirs=True,
    )
